In [55]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [56]:
## Load the dataset
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [57]:
## Preposcess the data
## Drop irrelevent columns

data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [58]:
## Encode categorical variables
## using LabelEncoder for Gender label_encoder_gender.fit_transform LabelEncoder()

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data 

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [59]:
## One-hot encode the 'Geography' column

from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder 


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [60]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [61]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [62]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [63]:
## Combine one hot encoded columns with the original data 
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [64]:
## save the encoders and scaler

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)



In [65]:
## Divide the dataset into independent and dependent features
 
X = data.drop('Exited', axis=1)
Y = data['Exited']

## Split the data in traning and testing sets
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

## Scale these feature 
scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [66]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]], shape=(8000, 12))

In [67]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [68]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


## ANN Implementation

In [69]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [70]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [71]:
## Build our ANN model

model=Sequential(
    [
        
        Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ## HL1 Connected with input layer
        Dense(32, activation='relu'), ## HL2
        Dense(1, activation='sigmoid') ## Output layer
    ]
)

c:\Users\AKSHANSH\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [72]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [73]:

import tensorflow


opt=tensorflow.keras.optimizers.Adam(learning_rate=0.001)
loss=tensorflow.keras.losses.BinaryCrossentropy()

In [74]:
## Compile the model
model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

In [75]:
## Set up the Teansorboard 
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callbacks = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [76]:
## Set up Early stopping 
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [77]:
%pip install tensorboard

Note: you may need to restart the kernel to use updated packages.


In [78]:
### Training the model
history=model.fit(
    X_train, Y_train, validation_data=(X_test, Y_test), epochs=100,
    callbacks=[tensorflow_callbacks, early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8052 - loss: 0.4559 - val_accuracy: 0.8290 - val_loss: 0.3991
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8406 - loss: 0.3896 - val_accuracy: 0.8545 - val_loss: 0.3579
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8544 - loss: 0.3570 - val_accuracy: 0.8570 - val_loss: 0.3519
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8600 - loss: 0.3457 - val_accuracy: 0.8600 - val_loss: 0.3437
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8599 - loss: 0.3385 - val_accuracy: 0.8580 - val_loss: 0.3477
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8602 - loss: 0.3354 - val_accuracy: 0.8595 - val_loss: 0.3432
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8630 - loss: 0.3321 - val_accuracy: 0.8600 - val_loss: 0.3426
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8640 - loss: 0.3290 - val_accu

In [79]:
model.save('model.h5')

In [80]:
%pip install tensorboard

Note: you may need to restart the kernel to use updated packages.


In [81]:
import sys
import shutil

print("Python executable:", sys.executable)
print("TensorBoard executable:", shutil.which("tensorboard"))

Python executable: c:\Users\AKSHANSH\anaconda3\python.exe
TensorBoard executable: None


In [82]:
%load_ext tensorboard
%tensorboard --logdir ./logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 30504), started 1 day, 22:19:39 ago. (Use '!kill 30504' to kill it.)

In [83]:
import sys
!{sys.executable} -m pip install tensorboard

In [84]:
import sys
import shutil
import importlib.util

print("Python:", sys.executable)
print("TensorBoard module:", importlib.util.find_spec("tensorboard"))
print("TensorBoard executable:", shutil.which("tensorboard"))

Python: c:\Users\AKSHANSH\anaconda3\python.exe
TensorBoard module: ModuleSpec(name='tensorboard', loader=<_frozen_importlib_external.SourceFileLoader object at 0x0000028A5ECF8E30>, origin='c:\\Users\\AKSHANSH\\anaconda3\\Lib\\site-packages\\tensorboard\\__init__.py', submodule_search_locations=['c:\\Users\\AKSHANSH\\anaconda3\\Lib\\site-packages\\tensorboard'])
TensorBoard executable: None


In [85]:
import os
import sys
from pathlib import Path

scripts_dir = Path(sys.executable).parent / "Scripts"
tensorboard_exe = scripts_dir / "tensorboard.exe"

print("Executable exists:", tensorboard_exe.exists())
print("Executable path:", tensorboard_exe)

Executable exists: True
Executable path: c:\Users\AKSHANSH\anaconda3\Scripts\tensorboard.exe


In [86]:
%env TENSORBOARD_BINARY=C:\Users\AKSHANSH\anaconda3\Scripts\tensorboard.exe
%reload_ext tensorboard
%tensorboard --logdir ./logs

env: TENSORBOARD_BINARY=C:\Users\AKSHANSH\anaconda3\Scripts\tensorboard.exe


Reusing TensorBoard on port 6006 (pid 30504), started 1 day, 22:19:41 ago. (Use '!kill 30504' to kill it.)

In [87]:
## Load Tensorboard extenxion 

%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [88]:
%tensorboard --logdir logs/fit20261002-152022

Reusing TensorBoard on port 6007 (pid 23336), started 1 day, 22:18:22 ago. (Use '!kill 23336' to kill it.)

In [89]:
%tensorboard --logdir logs/fit20261002-152022

Reusing TensorBoard on port 6007 (pid 23336), started 1 day, 22:18:22 ago. (Use '!kill 23336' to kill it.)

In [90]:
%tensorboard --logdir logs/fit20261002-152022/train --port 6007

Reusing TensorBoard on port 6007 (pid 25468), started 0:53:06 ago. (Use '!kill 25468' to kill it.)

In [91]:
!taskkill /F /PID 23336

ERROR: The process "23336" not found.


In [92]:
%reload_ext tensorboard
%tensorboard --logdir "logs/fit20261002-152022/train" --port 6007

Reusing TensorBoard on port 6007 (pid 25468), started 0:53:07 ago. (Use '!kill 25468' to kill it.)

In [93]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

log_dir = "logs/fit20261002-152022/train"

ea = EventAccumulator(log_dir)
ea.Reload()

print("Available tags:")
print(ea.Tags())

Available tags:
{'images': [], 'audio': [], 'histograms': [], 'scalars': [], 'distributions': [], 'tensors': ['keras'], 'graph': False, 'meta_graph': False, 'run_metadata': []}


In [94]:
import tensorflow as tf
import tensorboard

print("TensorFlow:", tf.__version__)
print("TensorBoard:", tensorboard.__version__)

TensorFlow: 2.21.0
TensorBoard: 2.21.0


In [95]:

import os
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

base_dir = "logs"

for root, dirs, files in os.walk(base_dir):
    if any(f.startswith("events.out.tfevents") for f in files):
        print("\nLog folder:", root)

        ea = EventAccumulator(root)
        ea.Reload()

        print("Tags:", ea.Tags())



Log folder: logs\fit20261002-152022\train
Tags: {'images': [], 'audio': [], 'histograms': [], 'scalars': [], 'distributions': [], 'tensors': ['keras'], 'graph': False, 'meta_graph': False, 'run_metadata': []}

Log folder: logs\fit20261002-152705\train
Tags: {'images': [], 'audio': [], 'histograms': [], 'scalars': [], 'distributions': [], 'tensors': ['keras', 'epoch_accuracy', 'epoch_loss', 'epoch_learning_rate', 'sequential_1/dense_3/kernel/histogram', 'sequential_1/dense_3/bias/histogram', 'sequential_1/dense_4/kernel/histogram', 'sequential_1/dense_4/bias/histogram', 'sequential_1/dense_5/kernel/histogram', 'sequential_1/dense_5/bias/histogram'], 'graph': False, 'meta_graph': False, 'run_metadata': []}

Log folder: logs\fit20261002-152705\validation
Tags: {'images': [], 'audio': [], 'histograms': [], 'scalars': [], 'distributions': [], 'tensors': ['evaluation_accuracy_vs_iterations', 'evaluation_loss_vs_iterations', 'epoch_accuracy', 'epoch_loss'], 'graph': False, 'meta_graph': Fals

In [96]:

import tensorflow as tf
from datetime import datetime

log_dir = "logs/fit" + datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=0,
    write_graph=False
)

history = model.fit(
    X_train,
    Y_train,
    epochs=10,
    validation_split=0.2,
    callbacks=[tensorboard_callback]
)


Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8673 - loss: 0.3192 - val_accuracy: 0.8687 - val_loss: 0.3153
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8686 - loss: 0.3182 - val_accuracy: 0.8600 - val_loss: 0.3192
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8684 - loss: 0.3169 - val_accuracy: 0.8644 - val_loss: 0.3266
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8673 - loss: 0.3158 - val_accuracy: 0.8681 - val_loss: 0.3224
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8708 - loss: 0.3137 - val_accuracy: 0.8612 - val_loss: 0.3254
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8708 - loss: 0.3142 - val_accuracy: 0.8594 - val_loss: 0.3257
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8681 - loss: 0.3119 - val_accuracy: 0.8644 - val_loss: 0.3271
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8691 - loss: 0.3117 - val_accuracy: 0.

In [97]:
%tensorboard --logdir logs --port 6007

ERROR: Failed to launch TensorBoard (exited with 4294967295).
Contents of stderr:
I0000 00:00:1791103478.059587    5976 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791103482.078559    5976 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
E1004 14:14:46.657087  5976 program.py:300] TensorBoard could not bind to port 6007, it was already in use
ERROR: TensorBoard could not bind to port 6007, it was already in use

In [98]:
%tensorboard --logdir logs --port 6008

Reusing TensorBoard on port 6008 (pid 27076), started 0:43:44 ago. (Use '!kill 27076' to kill it.)

In [99]:
%tensorboard --logdir "logs/fit20261002-153609" --port 6009

Reusing TensorBoard on port 6009 (pid 2676), started 0:38:52 ago. (Use '!kill 2676' to kill it.)